# FIFA World Cup 2026: Squad Age Analysis

**Analytic question:** Is the mean average squad age of national teams at the 2026 FIFA World Cup significantly different from 27 years?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# In GitHub: dataset goes in datasets/ and this notebook goes in task4_squad_age/.
INPUT_FILE = Path('../datasets/Squad Standard Stats(1).xlsx')

# This fallback lets the notebook run in the current workspace too.
if not INPUT_FILE.exists():
    INPUT_FILE = Path('upload/Squad Standard Stats(1).xlsx')

RANDOM_SEED = 42
SAMPLE_SIZE = 36
BENCHMARK_AGE = 27


## 1. Data wrangling
The exported FBref spreadsheet has a first row containing grouped headings. Row 2 contains the real variable names, so `header=1` is used.

In [ ]:
raw = pd.read_excel(INPUT_FILE, header=1)

# Retain only the variables needed for this task.
squads = raw.iloc[:, :3].copy()
squads.columns = ['Squad', 'Number_of_Players', 'Average_Squad_Age']

# Remove country codes such as 'dz ' and ensure age values are numeric.
squads['Squad'] = squads['Squad'].str.replace(r'^[a-z]{2,3}\s+', '', regex=True)
squads['Number_of_Players'] = pd.to_numeric(squads['Number_of_Players'], errors='coerce')
squads['Average_Squad_Age'] = pd.to_numeric(squads['Average_Squad_Age'], errors='coerce')

# Remove missing and duplicate squad records.
squads = squads.dropna().drop_duplicates(subset='Squad').reset_index(drop=True)

print(f'Number of cleaned squad records: {len(squads)}')
print(f'Missing values: {squads.isna().sum().sum()}')
squads.head()

## 2. Data preparation and sampling
The population is all 48 national squads at the 2026 FIFA World Cup. A simple random sample of 36 squads, without replacement, is selected using a fixed random seed so the work is reproducible.

In [ ]:
sample = squads.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).sort_values('Squad')
ages = sample['Average_Squad_Age']

sample.head()

## 3. Descriptive statistics

In [ ]:
descriptive_statistics = pd.DataFrame({
    'Statistic': ['Sample size', 'Mean age', 'Median age', 'Standard deviation', 'Minimum age', 'Maximum age'],
    'Value': [
        len(ages),
        ages.mean(),
        ages.median(),
        ages.std(ddof=1),
        ages.min(),
        ages.max()
    ]
})
descriptive_statistics

## 4. 95% confidence interval
This interval estimates the plausible range for the population mean squad age.

In [ ]:
standard_error = stats.sem(ages)
ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=len(ages) - 1,
    loc=ages.mean(),
    scale=standard_error
)

print(f'95% confidence interval: ({ci_low:.2f}, {ci_high:.2f}) years')

## 5. One-sample t-test
- **H₀:** The mean squad age is 27 years.
- **H₁:** The mean squad age is different from 27 years.

The significance level is α = 0.05.

In [ ]:
t_statistic, p_value = stats.ttest_1samp(ages, popmean=BENCHMARK_AGE)

print(f'Sample mean: {ages.mean():.2f} years')
print(f't-statistic: {t_statistic:.3f}')
print(f'p-value: {p_value:.4f}')

if p_value < 0.05:
    print('Decision: Reject H₀. The mean squad age is significantly different from 27 years.')
else:
    print('Decision: Fail to reject H₀. There is insufficient evidence that the mean squad age differs from 27 years.')

## 6. Visualisation

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(ages, bins=8, color='#0F766E', edgecolor='white')
plt.axvline(ages.mean(), color='#1D4ED8', linewidth=2, label=f'Sample mean = {ages.mean():.2f}')
plt.axvline(BENCHMARK_AGE, color='#DC2626', linestyle='--', linewidth=2, label='Test value = 27')
plt.title('Distribution of Average Squad Age')
plt.xlabel('Average squad age (years)')
plt.ylabel('Number of squads')
plt.legend()
plt.show()